In [17]:
!pip install langgraph langchain langchain-openai langchain-community chromadb beautifulsoup4

In [18]:
import os
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from typing_extensions import TypedDict
from typing import List
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langgraph.graph import StateGraph, END


In [19]:

llm = HuggingFacePipeline.from_model_id(
    model_id='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    task='text-generation',
    pipeline_kwargs=dict(
        temperature=0.5,
        max_new_tokens=100
    )
)

model = ChatHuggingFace(llm=llm)
grader_llm = ChatHuggingFace(llm=llm)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [20]:
# Indexing

print("Loading and indexing documents...")
# Load documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
)
docs = loader.load()

# Split documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3}) # Retrieve top 3 chunks
print("...Indexing complete.")

Loading and indexing documents...
...Indexing complete.


In [21]:
class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: The user's original question
        documents: The list of documents retrieved from ChromaDB
        generation: The LLM's generated answer
        query_rewrite_status: A flag ('yes' or 'no') to track if we rewrote the query
    """
    question: str
    documents: List[str]
    generation: str
    query_rewrite_status: str


In [22]:
def retrieve_node(state):
    """
    Retrieves documents from ChromaDB based on the current question.
    """
    print("---NODE: RETRIEVE---")
    question = state["question"]
    documents = retriever.invoke(question)
    doc_strings = [doc.page_content for doc in documents]
    print(f"Retrieved {len(doc_strings)} documents.")
    return {"documents": doc_strings, "question": question}


In [23]:

def generate_node(state):
    """
    Generates an answer using the retrieved documents.
    """
    print("---NODE: GENERATE---")
    question = state["question"]
    documents = state["documents"]
    
    # Prompt for the generator LLM
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Answer the user's question based *only* on the following context: {context}"),
        ("user", "Question: {question}")
    ])
    
    # Chain
    rag_chain = prompt | llm | StrOutputParser()
    
    # Run
    generation = rag_chain.invoke({"context": "\n---\n".join(documents), "question": question})
    
    print("Generated answer.")
    return {"documents": documents, "question": question, "generation": generation}


In [24]:

def grade_retrieval_node(state):
    """
    Grades the relevance of the retrieved documents.
    It checks if the documents are relevant to the question.
    """
    print("---NODE: GRADE RETRIEVAL---")
    question = state["question"]
    documents = state["documents"]
    
    # Prompt for the grader LLM
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a grader. Your task is to assess the relevance of retrieved documents to a user's question.
        If the documents contain information that can answer the question, output 'yes'.
        If the documents are not relevant or do not contain enough information, output 'no'.
        Provide *only* 'yes' or 'no' as your response."""),
        ("user", "Retrieved Documents:\n\n{documents}\n\nUser Question: {question}")
    ])
    
    grader_chain = prompt | grader_llm | StrOutputParser()
    
    context_str = "\n---\n".join(documents)
    decision = grader_chain.invoke({"documents": context_str, "question": question})
    
    if "yes" in decision.lower():
        print("DECISION: Documents are RELEVANT.")
        return {"query_rewrite_status": "no"} # No rewrite needed
    else:
        print("DECISION: Documents are NOT RELEVANT.")
        return {"query_rewrite_status": "yes"} # Rewrite is needed


In [25]:

def rewrite_query_node(state):
    """
    Rewrites the user's question to improve retrieval.
    This is used if the initial retrieval was graded as "not relevant".
    """
    print("---NODE: REWRITE QUERY---")
    question = state["question"]
    
    # Prompt for the rewriter LLM
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a query rewriter. Your task is to rephrase the user's question
        to make it more specific and easier to find in a vector database.
        Do not answer the question, just provide a better version of it."""),
        ("user", "Original Question: {question}")
    ])
    
    rewriter_chain = prompt | llm | StrOutputParser()
    
    new_question = rewriter_chain.invoke({"question": question})
    
    print(f"Rewritten question: {new_question}")
    return {"question": new_question} # Overwrites the original question in the state



In [26]:
print("Building the graph...")

workflow = StateGraph(GraphState)

# Add the nodes
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("grade_retrieval", grade_retrieval_node)
workflow.add_node("rewrite_query", rewrite_query_node)
workflow.add_node("generate", generate_node)

# Set the entry point
workflow.set_entry_point("retrieve")

# Add standard edges
workflow.add_edge("retrieve", "grade_retrieval")
workflow.add_edge("rewrite_query", "retrieve") # This creates the cycle!
workflow.add_edge("generate", END) # The graph finishes after generation

# Add the all-important CONDITIONAL edge
workflow.add_conditional_edges(
    "grade_retrieval",  # Start node
    
    # The condition function (lambda) checks the state
    lambda state: state["query_rewrite_status"],
    
    # The map defines where to go based on the condition's output
    {
        "no": "generate",     # If 'no' (rewrite not needed), go to generate
        "yes": "rewrite_query" # If 'yes' (rewrite is needed), go to rewrite
    }
)

# Compile the graph into a runnable application
app = workflow.compile()
print("Graph compiled.")



Building the graph...
Graph compiled.


In [27]:

# Example 1: A good, clear question
print("\n--- Good Question ---")
inputs = {"question": "What are the main components of an autonomous agent system?"}
for output in app.stream(inputs, config={"recursion_limit": 5}):
    # stream() yields outputs from each node as it executes
    for key, value in output.items():
        print(f"Output from node '{key}':")
        # print(value, "\n---\n") # Uncomment for full verbose output
    print("\n")

print("\n--- FINAL ANSWER (Run 1) ---")
print(value["generation"])


# Example 2: A vague question that will likely trigger a rewrite
print("\n\n--- RUN 2: Vague Question ---")
inputs = {"question": "What's the main idea about the agents?"}
for output in app.stream(inputs, config={"recursion_limit": 5}):
    for key, value in output.items():
        print(f"Output from node '{key}':")
        # print(value, "\n---\n") # Uncomment for full verbose output
    print("\n")

print("\n--- FINAL ANSWER (Run 2) ---")
print(value["generation"])


--- Good Question ---
---NODE: RETRIEVE---
Retrieved 3 documents.
Output from node 'retrieve':


---NODE: GRADE RETRIEVAL---
DECISION: Documents are RELEVANT.
Output from node 'grade_retrieval':


---NODE: GENERATE---
Generated answer.
Output from node 'generate':



--- FINAL ANSWER (Run 1) ---
System: You are a helpful assistant. Answer the user's question based *only* on the following context: LLM Powered Autonomous Agents | Lil'Log







































Lil'Log

















|






Posts




Archive




Search




Tags




FAQ









      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


 


Table of Contents



Agent System Overview

Component One: Planning

Task Decomposition

Self-Reflection


Component Two: Memory

Types of Memory

Maximum Inner Product Search (MIPS)


Component Three: Tool Use

Case Studies

Scientific Discovery Agent

Generative Agents Simulation

Proof-of-Concept Examp